In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

: 

In [2]:
logon_data = pd.read_csv('/kaggle/input/cert-insider-threat-detection-research/logon.csv')

In [ ]:
email_data = pd.read_csv('/kaggle/input/cert-insider-threat-detection-research/email.csv')

In [4]:
file_data = pd.read_csv('/kaggle/input/cert-insider-threat-detection-research/file.csv')

In [5]:
device_data = pd.read_csv('/kaggle/input/cert-insider-threat-detection-research/device.csv')

In [6]:
psychometric_data = pd.read_csv('/kaggle/input/cert-insider-threat-detection-research/psychometric.csv')

In [7]:
http_data = pd.read_csv('/kaggle/input/cert-insider-threat-detection-research/http.csv',nrows=3000000)

In [8]:
logon_data['date'] = pd.to_datetime(logon_data['date'])
logon_data['day'] = logon_data['date'].dt.dayofweek 
logon_data['hour'] = logon_data['date'].dt.hour

In [9]:
logon_features = logon_data.groupby('user').agg(
    L1=('pc', 'nunique'),  # Unique computers used by the user
    L2=('activity', lambda x: (x == 'Logon').sum()),  # Total logon events
    L3=('activity', lambda x: (x == 'Logoff').sum()),  # Total logoff events
    L4=('date', lambda x: x.dt.hour.mode()[0]),  # Most common logon hour
    L5=('date', lambda x: x.dt.hour.mode()[0] if len(x.dt.hour.mode()) > 0 else -1),  # Most common logoff hour
    L6=('day', lambda x: (x >= 5).sum()),  # Logons on weekends
    L7=('day', lambda x: (x < 5).sum()),  # Logons on weekdays
    L8=('pc', lambda x: x.value_counts().idxmax()),  # Most used PC
    L9=('hour', lambda x: ((x < 6) | (x > 18)).sum()),  # Logons outside working hours
    L10=('hour', lambda x: (x >= 6).sum()),  # Logons during working hours
    L11=('pc', lambda x: (x.value_counts().max() / len(x))),  # Ratio of logons on most used PC
    L12=('date', lambda x: (x.dt.hour > 22).sum()),  # Logons during late-night hours
    L13=('date', lambda x: (x.dt.hour < 6).sum()),  # Logons during early-morning hours
).reset_index()

In [10]:
logon_features

,user,L1,L2,L3,L4,L5,L6,L7,L8,L9,L10,L11,L12,L13
0,AAB0162,1,355,356,7,7,0,711,PC-6599,13,711,1.0,0,0
1,AAB0398,1,356,356,6,6,0,712,PC-8719,0,712,1.0,0,0
2,AAC0610,1,665,387,17,17,5,1047,PC-1834,51,1016,1.0,6,36
3,AAC0668,1,356,356,17,17,0,712,PC-9677,0,712,1.0,0,0
4,AAC3270,1,356,356,8,8,0,712,PC-5225,0,712,1.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,ZVB2656,1,356,356,17,17,0,712,PC-6247,0,712,1.0,0,0
3996,ZVS1637,1,356,356,16,16,0,712,PC-6157,0,712,1.0,0,0
3997,ZWS3625,1,356,356,15,15,0,712,PC-0695,0,712,1.0,0,0
3998,ZXM3086,1,356,356,17,17,0,712,PC-8526,0,712,1.0,0,0


In [11]:
email_data['date'] = pd.to_datetime(email_data['date'])
email_data['day'] = email_data['date'].dt.dayofweek  
email_data['hour'] = email_data['date'].dt.hour 

In [12]:
email_data['attachments'] = pd.to_numeric(email_data['attachments'], errors='coerce').fillna(0).astype(int)

In [13]:
email_features = email_data.groupby('user').agg(
    E1=('to', 'nunique'),  # Unique recipients emailed
    E2=('cc', 'count'),  # Total CC count
    E3=('bcc', 'count'),  # Total BCC count
    E4=('size', 'mean'),  # Average email size
    E5=('attachments', 'sum'),  # Total emails with attachments
    E6=('to', lambda x: sum('@' in str(addr) for addr in x.astype(str))),  # Emails sent outside organization
    E7=('attachments', lambda x: x.gt(0).sum()),  # Emails with at least one attachment
    E8=('hour', lambda x: ((x < 6) | (x > 18)).sum()),  # Emails sent outside working hours
    E9=('pc', 'nunique'),  # Unique PCs used for sending emails
    E10=('to', lambda x: x.value_counts().idxmax() if not x.empty else 'Unknown'),  # Most frequent recipient
    E11=('size', 'sum'),  # Total email size sent by user
    E12=('to', lambda x: x.dropna().astype(str).str.contains('gmail.com|yahoo.com|outlook.com').sum()),  # Emails sent to personal domains
    E13=('content', lambda x: x.dropna().astype(str).str.count('confidential|password|secure').sum()),  # Keyword frequency in emails
    E14=('content', lambda x: x.dropna().astype(str).str.count(r'\bthank you\b|\bappreciate\b').sum()),  # Positive sentiment keywords
    E15=('content', lambda x: x.dropna().astype(str).str.count(r'\burgent\b|\bimportant\b').sum()),  # Negative sentiment keywords
    E16=('content', lambda x: x.dropna().astype(str).str.len().mean()),  # Average email content length
).reset_index()

In [14]:
email_features

,user,E1,E2,E3,E4,E5,E6,E7,E8,E9,E10,E11,E12,E13,E14,E15,E16
0,AAB0162,1283,1482,0,358384.964399,0,3146,0,0,1,Amos.Ahmed.Burch@dtaa.com,1127479098,108,76,0,37,526.666561
1,AAB0398,1328,1391,0,601762.155340,0,3811,0,0,1,Allegra.April.Bates@dtaa.com,2293315574,524,19,0,72,562.028864
2,AAC0610,646,452,0,543528.577947,0,1052,0,0,1,Cote.Ahmed@yahoo.com,571792064,235,12,0,20,546.061787
3,AAC0668,1426,1242,0,386918.075120,0,3115,0,0,1,August.Amal.Carter@dtaa.com,1205249804,108,12,0,45,525.447512
4,AAC3270,212,155,0,867295.677326,0,344,0,0,1,Alexis.Aretha.Castro@dtaa.com,298349713,55,8,0,20,549.991279
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,ZVB2656,604,443,0,291791.451923,0,1040,0,0,1,Zia.Velma.Baxter@dtaa.com,303463110,36,5,0,38,547.097115
3996,ZVS1637,581,448,0,332505.891242,0,1039,0,0,1,Zephania.Valentine.Shields@dtaa.com,345473621,65,7,0,6,547.482194
3997,ZWS3625,233,169,0,947690.745714,0,350,0,0,1,Zephania.Wesley.Stout@dtaa.com,331691761,5,1,0,8,531.202857
3998,ZXM3086,1404,1417,0,477291.852031,0,3102,0,0,1,Zeph.Xerxes.Madden@dtaa.com,1480559325,220,53,0,70,543.936493


In [15]:
http_data['date'] = pd.to_datetime(http_data['date'])
http_data['day'] = http_data['date'].dt.dayofweek 
http_data['hour'] = http_data['date'].dt.hour

In [16]:
http_features = http_data.groupby('user').agg(
    H1=('url', 'nunique'),  # Unique websites visited
    H2=('activity', 'count'),  # Total web activities
    H3=('hour', lambda x: ((x < 6) | (x > 18)).sum()),  # Web activity outside working hours
    H4=('day', lambda x: (x >= 5).sum()),  # Web activity on weekends
    H5=('url', lambda x: x.str.contains('social|facebook|twitter|instagram', na=False).sum()),  # Visits to social media
    H6=('url', lambda x: x.str.contains('shopping|amazon|ebay', na=False).sum()),  # Visits to shopping sites
    H7=('url', lambda x: x.str.contains('news|cnn|bbc|nytimes', na=False).sum()),  # Visits to news sites
    H8=('url', lambda x: x.str.contains('entertainment|youtube|netflix', na=False).sum()),  # Visits to entertainment sites
    H9=('url', lambda x: x.str.len().mean()),  # Average URL length
    H10=('content', lambda x: x.dropna().astype(str).str.count('download|file|torrent').sum()),  # Downloads detected
    H11=('content', lambda x: x.dropna().astype(str).str.count(r'\bsecure\b|\bpassword\b').sum()),  # Security-related terms in content
    H12=('content', lambda x: x.dropna().astype(str).str.len().mean()),  # Average content length
).reset_index()

In [19]:
http_features

,user,H1,H2,H3,H4,H5,H6,H7,H8,H9,H10,H11,H12
0,AAB0162,56,821,0,0,0,1,0,0,87.809988,0,0,568.627284
1,AAB0398,119,1010,0,0,0,0,58,51,91.859406,0,0,643.578218
2,AAC0610,63,252,0,0,0,4,4,0,86.210317,2,0,667.476190
3,AAC0668,75,840,0,0,0,0,38,2,96.010714,0,0,664.680952
4,AAC3270,25,89,0,0,0,0,3,0,97.337079,0,0,622.910112
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,ZVB2656,64,254,0,0,0,3,3,0,93.326772,6,0,625.515748
3996,ZVS1637,58,257,0,0,3,0,22,0,93.194553,0,0,591.961089
3997,ZWS3625,31,90,0,0,0,0,2,6,93.122222,0,0,673.511111
3998,ZXM3086,127,833,0,0,0,9,31,0,88.314526,9,11,600.763505


In [17]:
file_data['date'] = pd.to_datetime(file_data['date'])
file_data['day'] = file_data['date'].dt.dayofweek  
file_data['hour'] = file_data['date'].dt.hour

In [18]:
file_data['to_removable_media'] = file_data['to_removable_media'].astype(bool)
file_data['from_removable_media'] = file_data['from_removable_media'].astype(bool)

In [19]:
file_features = file_data.groupby('user').agg(
    F1=('filename', 'nunique'),  # Unique files accessed
    F2=('activity', 'count'),  # Total file activities
    F3=('to_removable_media', 'sum'),  # Files copied to removable media
    F4=('from_removable_media', 'sum'),  # Files accessed from removable media
    F5=('hour', lambda x: ((x < 6) | (x > 18)).sum()),  # File activity outside working hours
    F6=('day', lambda x: (x >= 5).sum()),  # File activity on weekends
    F7=('activity', lambda x: x.str.contains('delete', case=False, na=False).sum()),  # Files deleted
    F8=('activity', lambda x: x.str.contains('copy', case=False, na=False).sum()),  # Files copied
    F9=('activity', lambda x: x.str.contains('move', case=False, na=False).sum()),  # Files moved
    F10=('content', lambda x: x.dropna().astype(str).str.count('confidential|password|sensitive').sum()),  # Sensitive keywords in files
    F11=('content', lambda x: x.dropna().astype(str).str.len().mean()),  # Average file content length
).reset_index()

In [20]:
file_features

,user,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11
0,AAB0162,8,30,0,0,0,0,0,0,0,0,528.166667
1,AAB0398,7,36,0,0,0,0,0,0,0,0,456.888889
2,AAC0610,641,1851,604,1056,7,0,404,825,0,0,568.686656
3,AAC0668,8,62,0,0,0,0,0,0,0,0,556.225806
4,AAC3270,2,8,0,0,0,0,0,0,0,0,757.875000
...,...,...,...,...,...,...,...,...,...,...,...,...
3180,ZRM0694,235,590,167,306,18,167,123,221,0,0,547.027119
3181,ZSL2618,5,10,0,0,0,0,0,0,0,0,586.400000
3182,ZWS3625,3,6,0,0,0,0,0,0,0,0,432.000000
3183,ZXM3086,10,80,0,0,0,0,0,0,0,0,581.187500


In [21]:
device_data['date'] = pd.to_datetime(device_data['date'])
device_data['day'] = device_data['date'].dt.dayofweek
device_data['hour'] = device_data['date'].dt.hour

In [22]:
device_features = device_data.groupby('user').agg(
    D1=('pc', 'nunique'),  # Unique PCs where devices were connected
    D2=('activity', 'count'),  # Total device activities
    D3=('hour', lambda x: ((x < 6) | (x > 18)).sum()),  # Device activity outside working hours
    D4=('day', lambda x: (x >= 5).sum()),  # Device activity on weekends
    D5=('activity', lambda x: x.str.contains('connect', case=False, na=False).sum()),  # Device connections
    D6=('activity', lambda x: x.str.contains('disconnect', case=False, na=False).sum()),  # Device disconnections
    D7=('file_tree', lambda x: x.dropna().astype(str).str.count('confidential|password|sensitive').sum()),  # Sensitive keywords in file names
    D8=('file_tree', lambda x: x.dropna().astype(str).str.len().mean()),  # Average file path length
).reset_index()

In [23]:
device_features

,user,D1,D2,D3,D4,D5,D6,D7,D8
0,AAC0610,1,700,6,0,700,349,0,58.0
1,AAF0819,1,2056,0,0,2056,1024,0,36.0
2,AAP0352,1,1420,0,0,1420,707,0,47.0
3,AAP1919,1,6159,0,0,6159,3070,0,47.0
4,ABD3426,1,7979,82,1742,7979,3979,0,14.0
...,...,...,...,...,...,...,...,...,...
783,ZCR0558,1,257,6,0,257,128,0,25.0
784,ZDC2523,1,367,1,98,367,182,0,58.0
785,ZHF1006,1,5610,0,0,5610,2799,0,25.0
786,ZNB3313,1,785,0,0,785,392,0,47.0


In [24]:
psychometric_features = psychometric_data[['user_id', 'O', 'C', 'E', 'A', 'N']]

In [25]:
psychometric_features.rename(columns={"user_id": "user"}, inplace=True)

<ipython-input-25-f37d68a2daed>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  psychometric_features.rename(columns={"user_id": "user"}, inplace=True)


In [26]:
psychometric_features

,user,O,C,E,A,N
0,NFP2441,34,39,38,36,21
1,ADR1517,36,39,13,19,27
2,MWB4000,27,14,44,22,34
3,MLS2856,35,49,22,45,28
4,BTR2026,15,15,41,39,27
...,...,...,...,...,...,...
3995,AJS0616,33,18,38,44,33
3996,ZOM0746,43,48,16,34,29
3997,GJT1292,46,26,13,35,28
3998,BLD2227,39,44,19,43,30


# MERGE ALL FEATURES

In [27]:
combined_df = logon_features
combined_df = combined_df.merge(email_features, on='user', how='outer')
combined_df = combined_df.merge(http_features, on='user', how='outer')
combined_df = combined_df.merge(file_features, on='user', how='outer')
combined_df = combined_df.merge(device_features, on='user', how='outer')
combined_df = combined_df.merge(psychometric_features, on='user', how='outer')

In [28]:
combined_df.shape

(4000, 66)

In [29]:
combined_df.head()

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pan

,user,L1,L2,L3,L4,L5,L6,L7,L8,L9,...,D4,D5,D6,D7,D8,O,C,E,A,N
0,AAB0162,1,355,356,7,7,0,711,PC-6599,13,...,NaN,NaN,NaN,NaN,NaN,45,36,33,15,33
1,AAB0398,1,356,356,6,6,0,712,PC-8719,0,...,NaN,NaN,NaN,NaN,NaN,16,38,16,26,34
2,AAC0610,1,665,387,17,17,5,1047,PC-1834,51,...,0.0,700.0,349.0,0.0,58.0,46,35,50,42,31
3,AAC0668,1,356,356,17,17,0,712,PC-9677,0,...,NaN,NaN,NaN,NaN,NaN,38,48,37,38,37
4,AAC3270,1,356,356,8,8,0,712,PC-5225,0,...,NaN,NaN,NaN,NaN,NaN,26,37,42,45,30


# ASSIGNING LABELS

In [30]:
feature_columns = ['F8', 'D3', 'F7', 'D6', 'H5', 'L9', 'F3', 'E6', 'H10']
feature_means = combined_df[feature_columns].mean()

In [31]:
def assign_label(row):
    # Normal Users: If all feature values are below 90% of their mean
    if all(row[feature] <= feature_means[feature] * 0.9 for feature in feature_columns):
        return "Normal"
    # Intellectual Property Theft: Features exceed 120% of mean
    elif row['F8'] > feature_means['F8'] * 1.2 or row['D3'] > feature_means['D3'] * 1.2:
        return "Intellectual Property Theft"
    # IT Sabotage: Features exceed 120% of mean
    elif row['F7'] > feature_means['F7'] * 1.2 or row['D6'] > feature_means['D6'] * 1.2:
        return "IT Sabotage"
    # Unauthorized Access: Features exceed 120% of mean
    elif row['H5'] > feature_means['H5'] * 1.2 or row['L9'] > feature_means['L9'] * 1.2:
        return "Unauthorized Access"
    # Data Exfiltration: Features exceed 120% of mean
    elif row['F3'] > feature_means['F3'] * 1.2 or row['E6'] > feature_means['E6'] * 1.2:
        return "Data Exfiltration"
    else:
        return "Normal"

In [32]:
combined_df['Label'] = combined_df.apply(assign_label, axis=1)

In [33]:
combined_df['Label'].unique()

array(['Normal', 'Data Exfiltration', 'Intellectual Property Theft',
       'IT Sabotage', 'Unauthorized Access'], dtype=object)

In [34]:
combined_df['Label'].value_counts()

Label
Normal                         1958
Unauthorized Access             907
Data Exfiltration               557
Intellectual Property Theft     533
IT Sabotage                      45
Name: count, dtype: int64

# SEPERATE FEATURES AND TARGET VARIABLE

In [35]:
X = combined_df.drop(columns=['user', 'Label'], errors='ignore') 
y = combined_df['Label']

# ENCODING CATEGORICAL VALUES

In [36]:
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    for col in categorical_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

# HANDLING MISSING VALUES

In [37]:
X.fillna(X.median(), inplace=True)

# APPLYING RANDOM FOREST CLASSIFER

In [38]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

RandomForestClassifier(random_state=42)

#  FEATURE SELECTION (65 → 47 SELECTED) 

In [39]:
feature_importances = pd.Series(rf.feature_importances_, index=X.columns)
selected_features = feature_importances[feature_importances > 0.001].nlargest(47).index.tolist()

In [40]:
selected_features = [
    'L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9', 'L10',  # Logon features
    'E1', 'E2', 'E3', 'E4', 'E5', 'E8', 'E9', 'E10', 'E12',  # Email features
    'H1', 'H2', 'H3', 'H4', 'H7', 'H8', 'H9',  # Web access features
    'F1', 'F4', 'F5', 'F8', 'F10', 'F11',  # File access features
    'D1', 'D2', 'D3', 'D4', 'D5', 'D6',  # External device features
    'O', 'C', 'E', 'A', 'N'  # Psychometric features
]

In [41]:
selected_df = combined_df[['user'] + selected_features + ['Label']]

In [42]:
selected_df.shape

(4000, 45)

In [43]:
selected_df.head()

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pan

,user,L1,L2,L3,L4,L5,L6,L7,L8,L9,...,D3,D4,D5,D6,O,C,E,A,N,Label
0,AAB0162,1,355,356,7,7,0,711,PC-6599,13,...,NaN,NaN,NaN,NaN,45,36,33,15,33,Normal
1,AAB0398,1,356,356,6,6,0,712,PC-8719,0,...,NaN,NaN,NaN,NaN,16,38,16,26,34,Data Exfiltration
2,AAC0610,1,665,387,17,17,5,1047,PC-1834,51,...,6.0,0.0,700.0,349.0,46,35,50,42,31,Intellectual Property Theft
3,AAC0668,1,356,356,17,17,0,712,PC-9677,0,...,NaN,NaN,NaN,NaN,38,48,37,38,37,Normal
4,AAC3270,1,356,356,8,8,0,712,PC-5225,0,...,NaN,NaN,NaN,NaN,26,37,42,45,30,Normal


In [44]:
selected_df.isna().sum()

user        0
L1          0
L2          0
L3          0
L4          0
L5          0
L6          0
L7          0
L8          0
L9          0
L10         0
E1          0
E2          0
E3          0
E4          0
E5          0
E8          0
E9          0
E10         0
E12         0
H1          0
H2          0
H3          0
H4          0
H7          0
H8          0
H9          0
F1        815
F4        815
F5        815
F8        815
F10       815
F11       815
D1       3212
D2       3212
D3       3212
D4       3212
D5       3212
D6       3212
O           0
C           0
E           0
A           0
N           0
Label       0
dtype: int64

In [45]:
file_device_cols = ['F1', 'F4', 'F5', 'F8', 'F10', 'F11', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6']
selected_df[file_device_cols] = selected_df[file_device_cols].fillna(0)

<ipython-input-45-275e62e73d93>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_df[file_device_cols] = selected_df[file_device_cols].fillna(0)


In [46]:
selected_df.isna().sum()

user     0
L1       0
L2       0
L3       0
L4       0
L5       0
L6       0
L7       0
L8       0
L9       0
L10      0
E1       0
E2       0
E3       0
E4       0
E5       0
E8       0
E9       0
E10      0
E12      0
H1       0
H2       0
H3       0
H4       0
H7       0
H8       0
H9       0
F1       0
F4       0
F5       0
F8       0
F10      0
F11      0
D1       0
D2       0
D3       0
D4       0
D5       0
D6       0
O        0
C        0
E        0
A        0
N        0
Label    0
dtype: int64

# ENCODING CATEGORICAL VALUES

In [47]:
selected_df = selected_df.drop(columns=['user'], errors='ignore')

In [48]:
label_encoders = {}
for col in ['L8', 'E10', 'Label']:
    le = LabelEncoder()
    selected_df[col] = le.fit_transform(selected_df[col])

# STANDARDIZING THE VALUES

In [49]:
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(selected_df.drop(columns=['Label']))

In [50]:
processed_df = pd.DataFrame(scaled_features, columns=selected_df.columns[:-1])
processed_df['Label'] = selected_df['Label']

In [51]:
def apply_pca(X_train, variance_retained=0.95):
    pca = PCA(n_components=variance_retained)
    X_train_pca = pca.fit_transform(X_train)
    return X_train_pca, pca

In [52]:
def build_generator(input_dim, feature_dim):
    input_layer = Input(shape=(input_dim,))
    x = Dense(32)(input_layer)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(64)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(64)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(128)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(512)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(1024)(x)
    x = LeakyReLU(alpha=0.2)(x)
    output_layer = Dense(feature_dim, activation='linear')(x)
    model = Model(input_layer, output_layer, name="Generator")
    return model

In [53]:
def build_generator(input_dim, feature_dim):
    input_layer = Input(shape=(input_dim,))
    
    # Wider network with proper initialization and normalization
    x = Dense(256, kernel_initializer='he_normal')(input_layer)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    x = Dense(512, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    x = Dense(1024, kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    # Final layer with tanh activation for bounded output
    output_layer = Dense(feature_dim, activation='tanh')(x)
    
    model = Model(input_layer, output_layer, name="Generator")
    return model

In [56]:
def build_discriminator(feature_dim):
    input_layer = Input(shape=(feature_dim,))
    x = Dense(1024)(input_layer)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(512)(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dense(128)(x)
    x = LeakyReLU(alpha=0.2)(x)
    validity = Dense(1, activation='sigmoid', name='Validity')(x)
    classification = Dense(5, activation='softmax', name='ClassLabel')(x)
    model = Model(input_layer, [validity, classification], name="Discriminator")
    return model

In [54]:
def build_discriminator(feature_dim, num_classes=5):
    input_layer = Input(shape=(feature_dim,))
    
    # Wider network with proper initialization and dropout
    x = Dense(1024, kernel_initializer='he_normal')(input_layer)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x)
    
    x = Dense(512, kernel_initializer='he_normal')(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x)
    
    x = Dense(256, kernel_initializer='he_normal')(x)
    x = LeakyReLU(alpha=0.2)(x)
    x = Dropout(0.3)(x)
    
    # Separate outputs for validity and class label
    validity = Dense(1, activation='sigmoid', name='Validity')(x)
    classification = Dense(num_classes, activation='softmax', name='ClassLabel')(x)
    
    model = Model(input_layer, [validity, classification], name="Discriminator")
    return model

In [55]:
def spca_loss(real_data, fake_data):
    # Convert to numpy for PCA
    real_data_np = real_data.numpy() if isinstance(real_data, tf.Tensor) else real_data
    fake_data_np = fake_data.numpy() if isinstance(fake_data, tf.Tensor) else fake_data
    
    # Fit PCA to get the loadings matrices
    n_components = min(3, real_data_np.shape[1])  # Ensure n_components doesn't exceed data dimensions
    
    pca_real = PCA(n_components=n_components)
    pca_real.fit(real_data_np)
    loadings_real = tf.constant(pca_real.components_.T, dtype=tf.float32)  # L matrix
    
    pca_fake = PCA(n_components=n_components)
    pca_fake.fit(fake_data_np)
    loadings_fake = tf.constant(pca_fake.components_.T, dtype=tf.float32)  # M matrix
    
    # Calculate S_PCA(A, B) = trace(LM^T ML^T)
    lmt = tf.matmul(loadings_real, tf.transpose(loadings_fake))
    mlt = tf.matmul(loadings_fake, tf.transpose(loadings_real))
    similarity = tf.linalg.trace(tf.matmul(lmt, mlt))
    
    # Return n_components - similarity as loss (to minimize)
    return tf.cast(n_components, tf.float32) - similarity


In [61]:
def train_spcagan(generator, discriminator, data, labels, epochs=1000, batch_size=64, latent_dim=100, save_interval=100):
    # Use higher learning rate for better convergence
    d_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5)
    g_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5)

    discriminator.compile(
        loss=['binary_crossentropy', 'sparse_categorical_crossentropy'], 
        optimizer=d_optimizer
    )
    
    noise_input = Input(shape=(latent_dim,))
    label_input = Input(shape=(1,), dtype=tf.int32)
    gen_output = generator(noise_input)
    
    # Freeze discriminator weights during generator training
    discriminator.trainable = False
    validity, class_prediction = discriminator(gen_output)
    combined = Model([noise_input, label_input], [validity, class_prediction])
    combined.compile(
        loss=['binary_crossentropy', 'sparse_categorical_crossentropy'], 
        optimizer=g_optimizer
    )
    
    labels = labels.to_numpy() if not isinstance(labels, np.ndarray) else labels
    
    for epoch in range(epochs):
        idx = np.random.randint(0, data.shape[0], batch_size)
        real_data, real_labels = data[idx], labels[idx]
        
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        fake_data = generator.predict(noise)
        fake_labels = np.random.randint(0, np.max(labels) + 1, batch_size)
        
        # Train discriminator
        discriminator.trainable = True
        for _ in range(3):  # Train discriminator multiple times
            d_loss_real = discriminator.train_on_batch(real_data, [np.ones((batch_size, 1)), real_labels])
            d_loss_fake = discriminator.train_on_batch(fake_data, [np.zeros((batch_size, 1)), fake_labels])
        
        # Train generator
        discriminator.trainable = False
        for _ in range(2):
            noise = np.random.normal(0, 1, (batch_size, latent_dim))
            
            # Generate fake data for SPCA calculation
            fake_data_for_spca = generator.predict(noise)
            
            # Compute SPCA loss
            spca_reg_loss = spca_loss(real_data, fake_data_for_spca)
            
            # Add SPCA loss to generator loss using custom training step
            g_loss = combined.train_on_batch(
                [noise, fake_labels], 
                [np.ones((batch_size, 1)), fake_labels]
            )
            
            # Apply SPCA regularization directly through optimizer
            with tf.GradientTape() as tape:
                fake_data_tensor = generator(tf.constant(noise, dtype=tf.float32))
                spca_loss_value = spca_loss(real_data, fake_data_tensor)
            
            # Get gradients and apply them with a weight of 0.1
            spca_gradients = tape.gradient(spca_loss_value, generator.trainable_weights)
            
            # Filter out None gradients and scale the rest
            spca_gradients_scaled = [(0.1 * g) if g is not None else g for g in spca_gradients]
            
            # Only apply gradients for non-None values
            grads_and_vars = [(g, v) for g, v in zip(spca_gradients_scaled, generator.trainable_weights) if g is not None]
            
            if grads_and_vars:  # Only apply if there are valid gradients
                g_optimizer.apply_gradients(grads_and_vars)
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}: D Loss Real={d_loss_real[0]:.4f}, D Loss Fake={d_loss_fake[0]:.4f}, G Loss={g_loss[0]:.4f}, SPCA Loss={spca_reg_loss:.4f}")

    return generator

In [62]:
X = processed_df.drop(columns=['Label'])  
y = processed_df['Label']

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [64]:
latent_dim = 100
feature_dim = 47
X_train_pca, pca_model = apply_pca(X_train)
generator = build_generator(latent_dim, X_train_pca.shape[1])
discriminator = build_discriminator(X_train_pca.shape[1])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [ ]:
trained_generator = train_spcagan(generator, discriminator, X_train_pca, y_train)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step  
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Epoch 0: D Loss Real=2.8074, D Loss Fake=2.7076, G Loss=2.7076, SPCA Loss=-0.0000
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
2/2 ━